# Milestone 2 — RAG Exploration

This notebook explores how Retrieval-Augmented Generation (RAG) works by combining our hybrid retrieval pipeline with Claude to synthesize natural language answers from retrieved product data.

**Pipeline:**
1. User submits a natural language query
2. Hybrid retrieval (BM25 + FAISS + RRF) fetches the top-k relevant products
3. Retrieved products are bundled into a prompt as context
4. Claude reads the context and generates a grounded answer


## Setup

In [1]:
import sys
from pathlib import Path

# Ensure project root is on path
ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.hybrid import retrieve_with_expansion
from src.query_expansion import generate_answer
from src.semantic import SemanticRetriever
from src import bm25 as bm25_mod

## Load Indices

In [2]:
# Load FAISS semantic index
index_dir = ROOT / "data" / "context_store" / "faiss_index"
semantic = SemanticRetriever(index_dir=index_dir)
semantic.load()
print("Semantic index loaded")

# Load BM25 index
bm25 = bm25_mod.load_bm25_retriever(bm25_mod.BM25_INDEX_PATH)
print("BM25 index loaded")

Semantic index loaded
BM25 index loaded


## Step 1 — Retrieval

Retrieve the top-k products for a query using hybrid search (BM25 + FAISS + RRF).

In [3]:
query = "what do reviewers say about drift issues on Switch controllers"

result = retrieve_with_expansion(
    query,
    mode="hybrid",
    expand=False,  # skip expansion for clarity
    top_k=5,
    semantic=semantic,
    bm25=bm25,
)

print(f"Query: {result.original_query}")
print(f"Hits returned: {len(result.hits)}\n")

for hit in result.hits:
    title = hit.get("product_title", "Unknown")
    rating = hit.get("average_rating", "N/A")
    score = hit.get("score", 0)
    print(f"  [{hit['rank']}] {title} | Rating: {rating} | RRF score: {score:.4f}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: what do reviewers say about drift issues on Switch controllers
Hits returned: 5

  [1] Joy Con for N-Switch/Switch Lite, FOCOLABU Switch Joy Pad Controller with Wake-up Function, Turbo, Motion Control & Dual Vibration, Gamepad Joystick Replacement for N-Switch Console | Rating: 3.6 | RRF score: 0.0164
  [2] DOYOKY Switch Controller, Switch Controller Compatible with Switch/Switch Lite, Wireless Gamepad with 7 LED Colors/Motion Control/Dual Vibration/Turbo | Rating: 4.4 | RRF score: 0.0164
  [3] DoinMaster Switch Ergonomic Controller for Nintendo Switch Handheld Mode, Ergonomic Controller, Compatible with All Games of Switch | Rating: 3.4 | RRF score: 0.0161
  [4] Joycon Joystick Replacement 4 Pack for Fix Drift Nintendo Switch Joy-Con Controller & Switch Lite Joystick Replacement Left/Right Analog Thumb Stick, Metal Latch, Include Y1.5 Screwdrivers | Rating: 4.4 | RRF score: 0.0161
  [5] Y Team Switch Pro Controller for Switch Lite, Switch Remote Gamepad Support Wake Up Console,

## Step 2 — Inspect Retrieved Context

The `content` field contains the `text_faiss` passage that was embedded and retrieved. This is the text Claude will read.

In [4]:
for hit in result.hits:
    title = hit.get("product_title", "Unknown")
    content = (hit.get("content") or "")[:400]
    print(f"--- {title}")
    print(content)
    print()

--- Joy Con for N-Switch/Switch Lite, FOCOLABU Switch Joy Pad Controller with Wake-up Function, Turbo, Motion Control & Dual Vibration, Gamepad Joystick Replacement for N-Switch Console
Product: Joy Con for N-Switch/Switch Lite, FOCOLABU Switch Joy Pad Controller with Wake-up Function, Turbo, Motion Control & Dual Vibration, Gamepad Joystick Replacement for N-Switch Console. Sold by FOCOLABU. Categories: Video Games Legacy Systems Nintendo Systems Nintendo DS Consoles. Features: no features listed. Rating: 3.6 from 39 ratings. Price: unknown. Customer reviews: [[VIDEOID:7d67ffe31

--- DOYOKY Switch Controller, Switch Controller Compatible with Switch/Switch Lite, Wireless Gamepad with 7 LED Colors/Motion Control/Dual Vibration/Turbo
Product: DOYOKY Switch Controller, Switch Controller Compatible with Switch/Switch Lite, Wireless Gamepad with 7 LED Colors/Motion Control/Dual Vibration/Turbo. Sold by binbok. Categories: Video Games Nintendo Switch Accessories Controllers. Features: 【Colo

## Step 3 — Build the RAG Prompt

Bundle the query and retrieved products into a prompt for Claude.

In [5]:
context_parts = []
for i, r in enumerate(result.hits, 1):
    title = r.get("product_title") or "Unknown product"
    rating = r.get("average_rating")
    rating_str = f" (rated {rating:.1f}/5)" if rating is not None else ""
    text = (r.get("content") or r.get("review_texts") or "").strip()
    snippet = text[:500] + ("…" if len(text) > 500 else "")
    context_parts.append(f"{i}. {title}{rating_str}\n{snippet}")

context = "\n\n".join(context_parts)

prompt = (
    f"User query: {query}\n\n"
    f"Top retrieved products:\n{context}\n\n"
    "Based only on the products above, answer the user's query in 2-3 sentences."
)

print(prompt)

User query: what do reviewers say about drift issues on Switch controllers

Top retrieved products:
1. Joy Con for N-Switch/Switch Lite, FOCOLABU Switch Joy Pad Controller with Wake-up Function, Turbo, Motion Control & Dual Vibration, Gamepad Joystick Replacement for N-Switch Console (rated 3.6/5)
Product: Joy Con for N-Switch/Switch Lite, FOCOLABU Switch Joy Pad Controller with Wake-up Function, Turbo, Motion Control & Dual Vibration, Gamepad Joystick Replacement for N-Switch Console. Sold by FOCOLABU. Categories: Video Games Legacy Systems Nintendo Systems Nintendo DS Consoles. Features: no features listed. Rating: 3.6 from 39 ratings. Price: unknown. Customer reviews: [[VIDEOID:7d67ffe3140613d726f24bfc1eae09ae]] I’ve tried a few different joy cons when my original controllers started t…

2. DOYOKY Switch Controller, Switch Controller Compatible with Switch/Switch Lite, Wireless Gamepad with 7 LED Colors/Motion Control/Dual Vibration/Turbo (rated 4.4/5)
Product: DOYOKY Switch Control

## Step 4 — Generate Answer with Claude

Send the bundled prompt to Claude and display the synthesized answer.

> Requires `ANTHROPIC_API_KEY` set in `.env`

In [6]:
answer = generate_answer(query, result.hits)
print("Claude's Answer:")
print("-" * 60)
print(answer)

Claude's Answer:
------------------------------------------------------------
Based on reviewer feedback, drift issues are a common concern with Switch controllers. The DoinMaster Ergonomic Controller review mentions that "if joycons get the drift problem. These really get it," suggesting drift is a widespread issue affecting many users. If you're experiencing drift with your original Joy-Cons, third-party replacement controllers like the FOCOLABU and DOYOKY options are popular alternatives, with the DOYOKY model rated 4.4/5 stars.


## Step 5 — Compare: Retrieval Only vs RAG

Without RAG, the user gets a list of products. With RAG, Claude synthesizes a direct answer from those products.

| Approach | Output |
|---|---|
| Retrieval only | List of top-k product titles + review snippets |
| RAG | A 2-3 sentence answer grounded in retrieved reviews |

In [7]:
# Try a second query to see how RAG handles different query types
query2 = "best open world RPG under $30 with good story"

result2 = retrieve_with_expansion(
    query2,
    mode="hybrid",
    expand=False,
    top_k=5,
    semantic=semantic,
    bm25=bm25,
)

print("Retrieved products:")
for hit in result2.hits:
    print(f"  [{hit['rank']}] {hit.get('product_title', 'Unknown')}")

print("\nClaude's Answer:")
print("-" * 60)
print(generate_answer(query2, result2.hits))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved products:
  [1] The Book of Unwritten Tales
  [2] Regions of Ruin - Nintendo Switch
  [3] The Witcher Enhanced Edition JC
  [4] To The Moon (Switch Limited Run #97) - Nintendo Switch
  [5] Advanced Dungeons & Dragons: Masterpiece Collection

Claude's Answer:
------------------------------------------------------------
Based on the available products, I'd recommend **Regions of Ruin for Nintendo Switch** ($29.99, 5.0/5 rating). It's a 2D open world RPG with town-building elements that progressively challenges you as you explore a vast continent—combining engaging gameplay with world-building. If you prefer a more traditional story-driven experience on PC, **The Witcher Enhanced Edition** (around $20-30) is excellent, featuring a rich narrative and character development set in a vibrant fantasy world inspired by Polish author Andrzej Sapkowski's works.


## Observations

- **RAG adds value for opinion queries** — e.g. "what do reviewers say about drift" requires synthesizing across multiple reviews. Retrieval alone returns a list; RAG gives a direct answer.
- **RAG is grounded** — Claude's answer is constrained to the retrieved products only, reducing hallucination risk.
- **RAG doesn't fix bad retrieval** — if the retriever returns irrelevant products, Claude's answer will also be off. Retrieval quality is the bottleneck.
- **Price constraints still fail** — "under $30" is not in the product metadata, so neither retrieval nor RAG can reliably enforce it.